### 148. 排序链表
给你链表的头结点 head ，请将其按 升序 排列并返回 排序后的链表 。

示例 1：
输入：head = [4,2,1,3]
输出：[1,2,3,4]

示例 2：
输入：head = [-1,5,3,4,0]
输出：[-1,0,3,4,5]


输入：head = []
输出：[]

#### 1.归并排序——分治
1. 找到链表的中间结点 head2 的前一个节点，并断开 head2 ​与其前一个节点的连接。这样我们就把原链表均分成了两段更短的链表。
2. 分治，递归调用 sortList，分别排序 head（只有前一半）和 head2。
3. 排序后，我们得到了两个有序链表，那么合并两个有序链表，得到排序后的链表，返回链表头节点。


In [ ]:
from typing import List, Optional
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

class Solution:
    def middleNode(self, head: Optional[ListNode]) -> Optional[ListNode]:
        # 1. 先将链表分为两半
        slow = fast = head
        while fast and fast.next:
            pre = slow # 记录slow的前一个节点
            slow = slow.next
            fast = fast.next.next
        pre.next = None # 将链表断开
        return slow

    # 2. 合并两个有序链表（双指针）
    def mergeTwoLists(self, l1: Optional[ListNode], l2: Optional[ListNode]) -> Optional[ListNode]:
        cur = dummy = ListNode() # 创建一个虚拟头节点，方便操作
        while l1 and l2:
            if l1.val < l2.val:
                cur.next = l1
                l1 = l1.next
            else:
                cur.next = l2
                l2 = l2.next
            cur = cur.next
        cur.next = l1 or l2 # 将剩余的链表连接到结果链表上
        return dummy.next

    # 3. 分治递归
    def sortList(self, head: Optional[ListNode]) -> Optional[ListNode]:
        # 分支，递归合并两个有序链表
        if not head or not head.next: # 没有两个节点，递归边界
            return head

        # 找到链表的中点，将链表分为两半
        head2 = self.middleNode(head)
        # 分治
        head = self.sortList(head) # 返回排序后的
        head2 = self.sortList(head2) 
        # 合并两个有序链表
        return self.mergeTwoLists(head, head2)

#### 2. 归并排序——迭代
- 方法 1 的归并是自顶向下计算，需要 O(logn) 的递归栈开销。
- 方法 2 将其改成自底向上计算，空间复杂度优化成 O(1)。

自底向上的意思是：
1. 首先，归并长度为 1 的子链表。例如 [4,2,1,3]，把第一个节点和第二个节点归并，第三个节点和第四个节点归并，得到 [2,4,1,3]。
2. 然后，归并长度为 2 的子链表。例如 [2,4,1,3]，把前两个节点和后两个节点归并，得到 [1,2,3,4]。
3. 然后，归并长度为 4 的子链表。
4. 依此类推，直到归并的长度大于等于链表长度为止，此时链表已经是有序的了。

具体步骤：
1. 遍历链表，获取链表长度 length。
2. 初始化步长 step=1。
3. 循环直到 step≥length。
4. 每轮循环，从链表头节点开始。
5. 分割出两段长为 step 的链表，合并，把合并后的链表插到新链表的末尾。重复该步骤，直到链表遍历完毕。
6. 把 step 扩大一倍。回到第 4 步。

作者：灵茶山艾府
链接：https://leetcode.cn/problems/sort-list/solutions/2993518/liang-chong-fang-fa-fen-zhi-die-dai-mo-k-caei/。

In [ ]:
class Solution:
    # 1. 获取链表长度
    def getLength(self, head: Optional[ListNode]) -> int:
        length = 0
        while head:
            length += 1
            head = head.next
        return length
    
    # 2. 分割链表
    # 如果链表长度 <= size，不做任何操作，返回空节点
    # 如果链表长度 > size，把链表的前 size 个节点分割出来（断开连接），并返回剩余链表的头节点
    def splitList(self, head: Optional[ListNode], size: int) -> Optional[ListNode]:
        # 先找到 next_head 的前一个节点
        cur = head
        for _ in range(size - 1): # 找到第 size 个节点
            if not cur: # 如果链表长度不足 size，直接返回空节点
                break
            cur = cur.next
        # 如果链表长度不足 size，直接返回空节点
        if cur is None or cur.next is None:
            return None
        
        next_head = cur.next # 记录剩余链表的头节点
        cur.next = None # 将链表断开
        return next_head
    
    # 3. 合并两个有序链表（双指针）
    def mergeTwoLists(self, l1: Optional[ListNode], l2: Optional[ListNode]) -> Optional[ListNode]:
        cur = dummy = ListNode() # 创建一个虚拟头节点，方便操作
        while l1 and l2:
            if l1.val < l2.val:
                cur.next = l1
                l1 = l1.next
            else:
                cur.next = l2
                l2 = l2.next
            cur = cur.next
        cur.next = l1 or l2 # 将剩余的链表连接到结果链表上
        while cur.next: # 迭代
            cur = cur.next
        # 循环结束后，cur 是合并后的链表的尾节点
        return dummy.next, cur
    
    # 4. 迭代合并
    def merge(self, head: Optional[ListNode]) -> Optional[ListNode]:
        length = self.getLength(head)  # 获取链表长度
        dummy = ListNode(next=head)  # 创建一个虚拟头节点，方便操作
        step = 1  # 初始步长为 1
        while step < length:  # 迭代
            
            new_list_tail = dummy  # 新链表的末尾
            cur = dummy.next  # 当前链表的头节点
            
            while cur:  # 遍历当前链表
                head1 = cur  # 第一个链表的头节点
                head2 = self.splitList(cur, step)  # 第二个链表的头节点
                cur = self.splitList(head2, step)  # 下一轮合并的头节点
                # 合并两段长为 step 的链表
                head, tail = self.mergeTwoLists(head1, head2)
                # 合并后的头节点 head，插到 new_list_tail 的后面
                new_list_tail.next = head
                new_list_tail = tail  # 更新 new_list_tail
            
            step *= 2
        return dummy.next